# RideFlow — 00 Provision

Create the demo schemas (LANDING, QUARANTINE, MARKETPLACE, PIPELINE). Warehouse + database are created by the deploy.


In [ ]:
# LakeLogic runs the medallion natively inside this container-runtime Notebook.
# NOTE: requires a LakeLogic build with the Snowflake-engine fixes
# (SnowflakeAdapter.set_shared_connection + TRY_CAST->TO_VARCHAR). Point LAKELOGIC_SPEC
# at the PyPI release that has them, or a git build of the repo/branch.
LAKELOGIC_SPEC = 'git+https://github.com/LakeLogic/LakeLogic@snowflake-engine-fixes'
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', LAKELOGIC_SPEC], check=True)


In [ ]:
import os, sys
# Resolve the repo root (folder containing domains_rideflow) from the Git-connected
# Workspace checkout, so runtime.* imports and relative contract paths both resolve.
d = os.getcwd()
while d != os.path.dirname(d) and not os.path.isdir(os.path.join(d, 'domains_rideflow')):
    d = os.path.dirname(d)
REPO = d; sys.path.insert(0, REPO); os.chdir(REPO)

DB  = 'RIDEFLOW_DEV_DEMO'
SYS = 'domains_rideflow/marketplace/rideflow/_system.yaml'
os.environ['SNOWFLAKE_DATABASE'] = DB   # contracts resolve {catalog} from this

from snowflake.snowpark.context import get_active_session
session = get_active_session()
conn = session.connection            # reuse THIS session (no password in the notebook)
from lakelogic.engines.snowflake import SnowflakeAdapter
SnowflakeAdapter.set_shared_connection(conn)
print('repo:', REPO)


In [ ]:
for s in ['LANDING','QUARANTINE','MARKETPLACE','PIPELINE']:
    session.sql(f'CREATE SCHEMA IF NOT EXISTS {DB}.{s}').collect()
print('schemas ready')
